In [2]:
import numpy as np 
import pandas as pd 
from pathlib import Path 
from collections import defaultdict
import json 


data_location = Path("/Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/raw/hmls")

visp = pd.read_csv(data_location / "VISPDAT" / "VISPDAT_with_serivices.csv")
vifsp = pd.read_csv(data_location / "VIFSPDAT" / "VIFSPDAT_with_serivices.csv")
tay = pd.read_csv(data_location / "TAYVISPDAT" / "TAYVISPDAT_with_serivices.csv")


visp = visp.rename(columns={
    "If you selected 'Other', please specify further details.": 
        "Where do you sleep most frequently? (Other - specify details)"
})

vifsp = vifsp.rename(columns={
    "If you selected 'Other', please specify further details.": 
        "Where do you and your family sleep most frequently? (Other - specify details)", 
    
    "8. f) Stayed one or more nights in a holding cell, jail, or prison, whether that was a short-term stay like the drunk tank, a longer stay for a more serious offense, or anything in between?": 
    "Has any family member stayed in jail, prison, or a holding cell?", 

    "35. Has any child in the family experienced abuse or trauma in the last 180 days?":
    "Has any child in the family experienced abuse or trauma in the last 180 days?"
})

tay = tay.rename(columns={
    "If you selected 'Other', please specify further details.": 
        "Where do you sleep most frequently? (Other - specify details)",

    "Have you spent one or more nights in a holding cell, jail, prison, or juvenile detention, regardless of the duration?": 
    "Have you spent nights in jail, prison, or detention?", 

    "Do you engage in risky behaviors (such as exchanging sex for money, food, drugs, or a place to stay; running drugs; having unprotected sex with strangers; sharing needles; etc.)?":
    "Do you engage in risky behaviors (e.g., exchanging sex, drugs, unsafe sex, sharing needles)?"
})

for df in [visp, vifsp, tay]:
    df["Client Uid"] = pd.to_numeric(df["Client Uid"], errors="coerce").astype("Int64")

In [3]:
def normalize_value(x):
    if pd.isna(x):
        return "Answer missing"
    if isinstance(x, str) and x.strip() == "":
        return "Answer missing"
    return str(x)

def get_question_columns(df):
    cols = list(df.columns)
    start_idx = cols.index("Start Date")
    total_idx = cols.index("GRAND TOTAL")
    return cols[start_idx + 1 : total_idx]

def get_band(score, dataset_name):
    score = pd.to_numeric(score, errors="coerce")
    if pd.isna(score):
        return None

    if dataset_name in {"VISPDAT", "TAYVISPDAT"}:
        if score <= 3:
            return "low"
        elif score <= 7:
            return "medium"
        else:
            return "high"

    elif dataset_name == "VIFSPDAT":
        if score <= 3:
            return "low"
        elif score <= 8:
            return "medium"
        else:
            return "high"

    else:
        raise ValueError(f"Unknown dataset: {dataset_name}")

In [ ]:

def export_household_jsons(df, dataset_name, base_path):
    dataset_dir = base_path / dataset_name

    for band in ["low", "medium", "high"]:
        (dataset_dir / band).mkdir(parents=True, exist_ok=True)

    question_cols = get_question_columns(df)
    seen_uids = defaultdict(int)
    skipped = 0

    for _, row in df.iterrows():
        uid = str(row["Client Uid"]).strip()
        band = get_band(row["GRAND TOTAL"], dataset_name)

        if band is None:
            skipped += 1
            continue

        seen_uids[uid] += 1
        if seen_uids[uid] == 1:
            filename = f"{uid}.json"
        else:
            filename = f"{uid}__{seen_uids[uid]}.json"

        payload = {
            col: normalize_value(row[col])
            for col in question_cols
        }

        outpath = dataset_dir / band / filename
        with open(outpath, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)

    print(f"{dataset_name}: wrote {len(df) - skipped} files, skipped {skipped} rows with missing/invalid GRAND TOTAL")



In [ ]:
export_household_jsons(visp, "VISPDAT", data_location)
export_household_jsons(vifsp, "VIFSPDAT", data_location)
export_household_jsons(tay, "TAYVISPDAT", data_location)

In [ ]:

def add_priority_band(df, dataset_name, score_col="GRAND TOTAL", new_col="priority_band"):
    scores = pd.to_numeric(df[score_col], errors="coerce")

    if dataset_name in {"VISPDAT", "TAYVISPDAT"}:
        df[new_col] = pd.Series(
            pd.cut(
                scores,
                bins=[-float("inf"), 3, 7, float("inf")],
                labels=["low", "medium", "high"]
            ),
            index=df.index
        ).astype("object")

    elif dataset_name == "VIFSPDAT":
        df[new_col] = pd.Series(
            pd.cut(
                scores,
                bins=[-float("inf"), 3, 8, float("inf")],
                labels=["low", "medium", "high"]
            ),
            index=df.index
        ).astype("object")

    else:
        raise ValueError(f"Unknown dataset_name: {dataset_name}")

    df.loc[scores.isna(), new_col] = pd.NA
    return df


visp = add_priority_band(visp, "VISPDAT")
vifsp = add_priority_band(vifsp, "VIFSPDAT")
tay = add_priority_band(tay, "TAYVISPDAT")


visp.to_csv(data_location / "VISPDAT" / "VISPDAT_with_serivices.csv", index=False)
vifsp.to_csv(data_location / "VIFSPDAT" / "VIFSPDAT_with_serivices.csv", index=False)
tay.to_csv(data_location / "TAYVISPDAT" / "TAYVISPDAT_with_serivices.csv", index=False)


### Make a tie-sheet 

In [ ]:
clean_location = Path("../../data/processed/pairwise_comparisons/hmls/")

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

data_location = Path("/Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/raw/hmls")

DATASETS = {
    "VISPDAT": data_location / "VISPDAT" / "VISPDAT_with_serivices.csv",
    "VIFSPDAT": data_location / "VIFSPDAT" / "VIFSPDAT_with_serivices.csv",
    "TAYVISPDAT": data_location / "TAYVISPDAT" / "TAYVISPDAT_with_serivices.csv",
}

BAND_ORDER = ["low", "medium", "high"]
POSITION_ORDER = ["lower", "middle", "upper"]


def load_dataset(path):
    df = pd.read_csv(path)
    df["Client Uid"] = pd.to_numeric(df["Client Uid"], errors="coerce").astype("Int64")
    df["GRAND TOTAL"] = pd.to_numeric(df["GRAND TOTAL"], errors="coerce")

    if "priority_band" not in df.columns:
        raise ValueError(f"{path} is missing 'priority_band'.")

    df["priority_band"] = df["priority_band"].astype("string").str.lower()
    df = df.dropna(subset=["Client Uid", "GRAND TOTAL", "priority_band"]).copy()
    return df




In [ ]:
def pick_lower_middle_upper(group):
    group = group.sort_values(
        by=["GRAND TOTAL", "Client Uid"],
        ascending=[True, True]
    ).reset_index(drop=True)

    n = len(group)
    if n < 3:
        raise ValueError(
            f"Band '{group['priority_band'].iloc[0]}' has only {n} rows; need at least 3."
        )

    idxs = [0, n // 2, n - 1]
    selected = group.iloc[idxs].copy()
    selected["within_band_position"] = POSITION_ORDER
    return selected


def select_representatives(df, dataset_name):
    pieces = []

    for band in BAND_ORDER:
        band_df = df[df["priority_band"] == band].copy()
        picked = pick_lower_middle_upper(band_df)
        picked["dataset_name"] = dataset_name
        pieces.append(picked)

    selected = pd.concat(pieces, ignore_index=True)

    selected = selected[
        ["dataset_name", "priority_band", "within_band_position", "Client Uid", "GRAND TOTAL"]
    ].copy()

    selected["priority_band"] = pd.Categorical(
        selected["priority_band"], categories=BAND_ORDER, ordered=True
    )
    selected["within_band_position"] = pd.Categorical(
        selected["within_band_position"], categories=POSITION_ORDER, ordered=True
    )

    selected = selected.sort_values(
        by=["priority_band", "within_band_position"]
    ).reset_index(drop=True)

    return selected


def run_selection_small():
    all_selected = {}

    for dataset_name, path in DATASETS.items():
        df = load_dataset(path)
        selected = select_representatives(df, dataset_name)

        out_path = clean_location / dataset_name / "selected_households.csv"
        selected.to_csv(out_path, index=False)

        print(f"\n{dataset_name}")
        print(selected.to_string(index=False))

        all_selected[dataset_name] = selected

    return all_selected


all_selected = run_selection_small()

In [15]:
clean_location = Path("/Users/gaurabpokharel/Library/CloudStorage/Box-Box/WashU-ICA-SFT-Working Folder - shared/Sandboxes/Gaurab Pokharel/VibeRank/processed/rank_centrality/hmls/")
def pick_stratified(group, n_per_band):
    group = group.sort_values(
        by=["GRAND TOTAL", "Client Uid"],
        ascending=[True, True],
    ).reset_index(drop=True)
    n = len(group)
    if n < n_per_band:
        raise ValueError(
            f"Band '{group['priority_band'].iloc[0]}' has only {n} rows; "
            f"need at least {n_per_band}."
        )
    idxs = np.linspace(0, n - 1, n_per_band, dtype=int)
    selected = group.iloc[idxs].copy()
    selected["within_band_position"] = list(range(1, n_per_band + 1))
    return selected


def select_representatives(df, dataset_name, n_per_band):
    pieces = []
    for band in BAND_ORDER:
        band_df = df[df["priority_band"] == band].copy()
        picked = pick_stratified(band_df, n_per_band)
        picked["dataset_name"] = dataset_name
        pieces.append(picked)
    selected = pd.concat(pieces, ignore_index=True)
    selected = selected[
        ["dataset_name", "priority_band", "within_band_position",
         "Client Uid", "GRAND TOTAL"]
    ].copy()
    selected["priority_band"] = pd.Categorical(
        selected["priority_band"], categories=BAND_ORDER, ordered=True
    )
    selected = selected.sort_values(
        by=["priority_band", "within_band_position"]
    ).reset_index(drop=True)
    return selected


def run_selection(n_per_band=10):
    all_selected = {}
    for dataset_name, path in DATASETS.items():
        df = load_dataset(path)
        selected = select_representatives(df, dataset_name, n_per_band)
        out_path = clean_location / dataset_name / "selected_households.csv"
        selected.to_csv(out_path, index=False)
        print(f"\n{dataset_name} ({n_per_band} per band, "
              f"{n_per_band * len(BAND_ORDER)} total)")
        print(selected.to_string(index=False))
        all_selected[dataset_name] = selected
    return all_selected

In [16]:
all_ = run_selection()


VISPDAT (10 per band, 30 total)
dataset_name priority_band  within_band_position  Client Uid  GRAND TOTAL
     VISPDAT           low                     1       51360            1
     VISPDAT           low                     2      341645            1
     VISPDAT           low                     3      211435            2
     VISPDAT           low                     4      262923            2
     VISPDAT           low                     5      327739            2
     VISPDAT           low                     6         685            3
     VISPDAT           low                     7       84305            3
     VISPDAT           low                     8      256880            3
     VISPDAT           low                     9      303448            3
     VISPDAT           low                    10      352916            3
     VISPDAT        medium                     1         505            4
     VISPDAT        medium                     2      266515            4
     